##  전담한 역할 및 문제 해결 과정

### 1. 프로젝트 정보 및 결과물 요약
* **프로젝트 기간:** 2025년 3월 ~ 2025년 9월 
* **팀원 구성**
* **참여 인원:** 4명 
  - 팀원 (최성용): 하드웨어 회로 설계, 센서 노드 원시 코드 작성
  - 팀원 (구민서): 외형 PETG 3D 케이스 설계 및 구조물 조립
  - 팀원 (이명은): ESP32-앱 간 블루투스 통신 기초 테스트 및 디버깅, 앱 표지 디자인
* **프로젝트 개요:** 안전모에 센서를 달아 작업자가 장비를 잘 썼는지, 현장에서 쓰러지거나 넘어지지 않았는지 블루투스로 받아와 확인하는 실시간 안전 모니터링 시스템임.
* **핵심 결과물:** 앱인벤터 툴이 가진 블루투스 연결 개수 한계를 하드웨어 중계기 아이디어로 풀었음. 스마트폰 화면 하나에 여러 명의 작업자 상태가 동시에 뜨도록 화면과 통신 구조를 완성함.

---

### 2. 내가 전담한 역할
* **모바일 관제 어플리케이션(앱인벤터) 구현:** 관리자가 스마트폰으로 현장 상황을 볼 수 있도록 표 형태의 화면을 디자인하고, 들어오는 데이터를 제 칸에 뿌려주는 블록 코딩 전체를 전담함.
* **무선 통신 연결 및 데이터 설계:** 메인 중계기 기기와 스마트폰 앱 사이에 주고받을 블루투스(BLE) 데이터 형태를 직접 정의하고 연동 테스트를 진행함.

---

### 3. 개발 과정 중 발생한 문제 및 해결 내용 (트러블 슈팅)

####  01. 여러 대의 기기를 앱에 동시에 연결할 수 없는 한계 해결
> **어려움**
> 현장 작업자가 여러 명이라 안전모마다 독립된 송신 기기가 들어가야 했음. 하지만 앱인벤터 구조상 스마트폰 하나에 여러 개의 블루투스 기기를 동시에 안정적으로 직접 연결하는 것이 불가능했음.

> **해결 내용**
> 스마트폰 앱에서 기기 여러 개를 직접 잡는 방식을 포기함. 작업자들의 기기 신호를 한곳으로 모아줄 **'메인 중계기(ESP32)'**를 중간에 하나 더 두는 방식을 냈음. 각 작업자의 기기가 보낸 신호를 중간 중계기가 다 받아낸 뒤, 스마트폰 앱에는 이 중계기 1대만 연결해서 데이터를 한 번에 밀어 넣어주는 통신 구조를 만들어 문제를 해결함.

####  02. 들어오는 데이터가 섞이고 뭉개지는 오류 해결
> **어려움**
> 중간 중계기 한 대가 여러 명의 데이터를 모아서 스마트폰 앱으로 한꺼번에 던지다 보니, 앱인벤터 쪽에서 이 데이터가 누구의 데이터인지 구분을 못 해 화면에 엉뚱한 값이 찍히거나 글자가 깨지는 현상이 생김.

> **해결 내용**
> 데이터를 보낼 때 무작정 보내지 않고 `[작업자 번호/현재 위치/장비 착용 상태]` 순서로 데이터 사이에 슬래시 나 쉼표 같은 약속된 기호를 넣어서 패킷을 묶어 보내도록 수정함. 앱인벤터 블록 코딩에서는 문자열 분리(Split) 블록을 사용해 들어온 데이터를 기호 기준으로 똑똑하게 쪼갠 뒤, 화면의 `작업자1`, `작업자2` 칸에 정확히 찾아 들어가도록 로직을 짜서 데이터가 섞이는 문제를 잡았음.

####  03. 쉴 새 없이 들어오는 신호로 인한 화면 멈춤 현상 해결
> **어려움**
> 여러 명의 데이터가 블루투스를 통해 실시간으로 너무 자주 들어오다 보니, 앱인벤터 화면이 버퍼링을 버티지 못하고 수시로 멈추거나 앱이 강제로 꺼지는 현상이 발생함.

> **해결 내용**
> 무조건 신호를 계속 쏘는 방식 대신, 주기를 1~2초 간격으로 조절하고 평소에는 조용하다가 작업자가 안전모를 벗거나 쓰러지는 등의 '상태 변화'가 생겼을 때만 즉시 신호를 쏘도록 송신 방식을 바꿨음. 데이터가 들어오는 통로의 부하를 줄여주어 앱이 끊김 없이 부드럽게 작동하도록 완성함.

####  04. 중계기(Master)의 다중 데이터 수신 병목 및 유실 현상 해결
> **어려움**
> 각 작업자의 기기(Slave)들이 중계기(Master) 한 대를 향해 동시에 블루투스 신호를 보내다 보니, 데이터가 같은 시간에 겹쳐서 들어올 때 신호가 충돌하여 일부 작업자의 상태 데이터가 앱으로 전달되지 못하고 사라지는(유실) 현상이 생김.

> **해결 내용**
> 중계기(ESP32) 펌웨어 코드를 직접 수정하여 통신 안정성을 높였음. 각 작업자의 기기들이 무작작 신호를 던지지 않고, 중계기가 `작업자1 신호 줘`, `작업자2 신호 줘` 하는 식으로 순서대로 신호를 요청하고 받아오는 수신 로직을 중계기 코드로 구현함. 또한, 들어온 데이터가 겹치더라도 안전하게 임시 보관했다가 순서대로 앱에 보내주도록 중계기 내부에 데이터 버퍼링 구조를 짜넣어 데이터가 중간에 날아가는 문제를 완벽히 해결함.


## 팀 프로젝트 개요 
* **참여 인원:** 4명 
  - 팀원 (최성용): 하드웨어 회로 설계, 센서 노드 원시 코드 작성
  - 팀원 (구민서): 외형 PETG 3D 케이스 설계 및 구조물 조립
  - 팀원 (이명은): ESP32-앱 간 블루투스 통신 기초 테스트 및 디버깅, 앱 표지 디자인